# Validating a Model Before Deployment

Pre-deployment validation is a gate check, not a training evaluation. It answers the question: "Is this model safe and fast enough to put in front of real users?" This notebook defines acceptance criteria, tests a trained model against them, and wraps everything in a `validate_model()` function that returns a clear pass/fail result.

**Learning objectives**
1. Distinguish pre-deployment validation from training-time evaluation.
2. Define acceptance criteria: accuracy threshold, latency budget, no NaN outputs.
3. Test a model against a held-out validation set, a latency benchmark, and malformed inputs.
4. Write a `validate_model()` function that returns a structured pass/fail report.


## 1  What is pre-deployment validation?

During training you optimize a loss function. Pre-deployment validation is different: it checks whether the trained model meets *business and operational requirements* before serving real users. You set a threshold, run the model against a held-out set it has never seen, and either deploy or reject. This is a one-shot gate, not a learning step.


In [1]:
# WHAT: build a train/validation/test split, train a pipeline, save the artifact.
# WHY: the validation set is reserved for the DEPLOYMENT GATE — a decision set the
# model never trained on, so the gate cannot be gamed by memorization.
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib
import time
import json

iris = load_iris()

# Three-way split: train / validation / test
# Validation is used only for the deployment gate — never during training
X_train, X_temp, y_train, y_temp = train_test_split(
    iris.data, iris.target, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Scaler + forest in one pipeline: the artifact under validation is the real thing.
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)

# Save to disk — the gate below validates the SAVED artifact, as production would.
MODEL_PATH = "/tmp/val_pipeline.joblib"
joblib.dump(pipeline, MODEL_PATH)

print(f"Training samples  : {len(X_train)}")
print(f"Validation samples: {len(X_val)}  (used for deployment gate)")
print(f"Test samples      : {len(X_test)}  (held out entirely)")
print(f"Training accuracy : {pipeline.score(X_train, y_train):.2%}")

Training samples  : 90
Validation samples: 30  (used for deployment gate)
Test samples      : 30  (held out entirely)
Training accuracy : 100.00%


## 2  Define acceptance criteria

Acceptance criteria are set before validation runs — not after seeing the results. Setting them after the fact is "p-hacking" for ML.


In [2]:
# WHAT: write down the acceptance criteria BEFORE looking at any validation result.
# WHY: criteria chosen after seeing the numbers always pass — fixing them upfront
# is what makes the gate honest (same idea as pre-registered experiments).
# These criteria are defined upfront — not tuned after seeing validation results
CRITERIA = {
    "min_accuracy": 0.90,        # Model must get >= 90% on the held-out validation set
    "max_p95_latency_ms": 100.0, # 95th-percentile single-prediction latency must be < 100 ms
    "allow_nan_output": False,   # Model must never return NaN predictions
    "n_latency_samples": 200,    # Number of single-sample predictions to benchmark
}

print("Acceptance criteria:")
for k, v in CRITERIA.items():
    print(f"  {k}: {v}")

Acceptance criteria:
  min_accuracy: 0.9
  max_p95_latency_ms: 100.0
  allow_nan_output: False
  n_latency_samples: 200


## 3  Accuracy gate — run against the validation set


In [3]:
# WHAT: load the saved artifact and apply the accuracy gate on the validation set.
# WHY: we test the file on disk, not the in-memory object — a corrupted or stale
# artifact should fail HERE, before it ever reaches users.
model = joblib.load(MODEL_PATH)

val_preds = model.predict(X_val)
val_accuracy = float(np.mean(val_preds == y_val))

# Compare against the pre-committed threshold and report PASS/FAIL explicitly.
accuracy_pass = val_accuracy >= CRITERIA["min_accuracy"]
print(f"Validation accuracy : {val_accuracy:.2%}")
print(f"Threshold           : {CRITERIA['min_accuracy']:.0%}")
print(f"Accuracy gate       : {'PASS' if accuracy_pass else 'FAIL'}")

Validation accuracy : 100.00%
Threshold           : 90%
Accuracy gate       : PASS


## 4  Latency gate — time 200 single-sample predictions

We predict one sample at a time (not a batch) because production usually receives individual requests. We measure p95 to catch tail latency — the slow cases that actually affect users.


In [4]:
# WHAT: time 200 single-sample predictions and apply the p95 latency gate.
# WHY: accuracy is not enough — a correct model that answers in 2 seconds still
# fails users. Single-sample calls mimic real-time API traffic, not batch scoring.
latencies_ms = []
n = CRITERIA["n_latency_samples"]

# Cycle through validation rows so timing is not dominated by one cached input.
for i in range(n):
    sample = X_val[i % len(X_val)].reshape(1, -1)
    t0 = time.perf_counter()
    model.predict(sample)
    latencies_ms.append((time.perf_counter() - t0) * 1000)

# p95 is the gate metric: 19 of 20 requests must beat the threshold.
lat = np.array(latencies_ms)
p95 = float(np.percentile(lat, 95))
latency_pass = p95 < CRITERIA["max_p95_latency_ms"]

print(f"Predictions timed : {n}")
print(f"  p50             : {np.percentile(lat, 50):.2f} ms")
print(f"  p95             : {p95:.2f} ms")
print(f"  p99             : {np.percentile(lat, 99):.2f} ms")
print(f"Latency threshold : {CRITERIA['max_p95_latency_ms']:.0f} ms (p95)")
print(f"Latency gate      : {'PASS' if latency_pass else 'FAIL'}")

Predictions timed : 200
  p50             : 2.50 ms
  p95             : 2.60 ms
  p99             : 2.67 ms
Latency threshold : 100 ms (p95)
Latency gate      : PASS


## 5  Input validation — test with malformed inputs

A robust model serving layer must not crash on bad input. Here we test how the pipeline handles inputs that should never reach a production model but sometimes do.


In [5]:
# WHAT: probe the model with malformed inputs (wrong shape, NaN, empty batch).
# WHY: production traffic WILL contain garbage — we document how the model fails so
# the API layer knows exactly which errors it must catch before calling predict.
from typing import List

# Wrapper that converts exceptions into printable strings instead of crashing the cell.
def safe_predict(model, X):
    """Try to predict; return an error string if it fails."""
    try:
        return model.predict(X)
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

# Wrong number of features
wrong_shape = np.array([[5.1, 3.5]])  # only 2 features instead of 4
result_a = safe_predict(model, wrong_shape)
print(f"Wrong shape (2 features): {result_a}")

# NaN values in input
nan_input = np.array([[5.1, np.nan, 1.4, 0.2]])
result_b = safe_predict(model, nan_input)
print(f"NaN in input            : {result_b}")

# Empty array
empty_input = np.array([]).reshape(0, 4)
result_c = safe_predict(model, empty_input)
print(f"Empty input             : {result_c}")

print("\nInput validation checks complete — errors above are expected and should be caught at the API layer.")

Wrong shape (2 features): ERROR: ValueError: X has 2 features, but StandardScaler is expecting 4 features as input.
NaN in input            : [0]
Empty input             : ERROR: ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by StandardScaler.

Input validation checks complete — errors above are expected and should be caught at the API layer.


## 6  Output validation — check shape, ranges, and NaN


In [6]:
# Valid predictions on the validation set
preds = model.predict(X_val)
probas = model.predict_proba(X_val)

# Output shape checks
assert preds.shape == (len(X_val),), f"Wrong output shape: {preds.shape}"
assert probas.shape == (len(X_val), 3), f"Wrong proba shape: {probas.shape}"

# Value range checks
assert not np.any(np.isnan(preds)), "NaN in predictions"
assert not np.any(np.isnan(probas)), "NaN in probability outputs"
assert np.all(probas >= 0) and np.all(probas <= 1), "Probabilities out of [0, 1]"

# Probabilities must sum to 1 per sample
row_sums = probas.sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-6), f"Probabilities don't sum to 1: {row_sums[:5]}"

# Predictions must be valid class indices
valid_classes = set(range(3))
assert set(preds).issubset(valid_classes), f"Unexpected class values: {set(preds)}"

no_nan_pass = not (np.any(np.isnan(preds)) or np.any(np.isnan(probas)))

print("Output shape  : PASS")
print("No NaN output : PASS")
print("Proba range   : PASS")
print("Proba sums    : PASS")
print("Valid classes : PASS")


Output shape  : PASS
No NaN output : PASS
Proba range   : PASS
Proba sums    : PASS
Valid classes : PASS


## 7  `validate_model()` — the deployment gate function


In [7]:
# WHAT: combine accuracy, latency, and NaN checks into one reusable gate function
# that returns a structured report.
# WHY: this is the notebook's payoff — a single validate_model() call a CI pipeline
# could run on every new artifact, with machine-readable pass/fail per check.
def validate_model(model, X_val, y_val, criteria):
    """
    Run all deployment validation checks and return a structured report.
    Returns a dict with 'passed' (bool) and 'checks' (per-check results).
    """
    report = {"passed": True, "checks": {}}

    # 1. Accuracy
    preds = model.predict(X_val)
    accuracy = float(np.mean(preds == y_val))
    acc_ok = accuracy >= criteria["min_accuracy"]
    report["checks"]["accuracy"] = {
        "value": round(accuracy, 4),
        "threshold": criteria["min_accuracy"],
        "passed": acc_ok
    }
    if not acc_ok:
        report["passed"] = False

    # 2. Latency (p95 of single-sample predictions)
    times_ms = []
    n = criteria["n_latency_samples"]
    for i in range(n):
        s = X_val[i % len(X_val)].reshape(1, -1)
        t0 = time.perf_counter()
        model.predict(s)
        times_ms.append((time.perf_counter() - t0) * 1000)
    p95 = float(np.percentile(times_ms, 95))
    lat_ok = p95 < criteria["max_p95_latency_ms"]
    report["checks"]["p95_latency_ms"] = {
        "value": round(p95, 3),
        "threshold": criteria["max_p95_latency_ms"],
        "passed": lat_ok
    }
    if not lat_ok:
        report["passed"] = False

    # 3. No NaN in outputs
    probas = model.predict_proba(X_val)
    no_nan = not (np.any(np.isnan(preds)) or np.any(np.isnan(probas)))
    report["checks"]["no_nan_output"] = {"passed": no_nan}
    if not no_nan:
        report["passed"] = False

    return report


# Run the full gate on our artifact and print the verdict a pipeline would act on.
result = validate_model(model, X_val, y_val, CRITERIA)

print(f"Deployment gate: {'APPROVED' if result['passed'] else 'REJECTED'}")
print("\nDetailed results:")
print(json.dumps(result, indent=2))

Deployment gate: APPROVED

Detailed results:
{
  "passed": true,
  "checks": {
    "accuracy": {
      "value": 1.0,
      "threshold": 0.9,
      "passed": true
    },
    "p95_latency_ms": {
      "value": 2.574,
      "threshold": 100.0,
      "passed": true
    },
    "no_nan_output": {
      "passed": true
    }
  }
}


## Summary

Pre-deployment validation is a formal gate, not an exploration. Key points:

| Check | What it catches |
|---|---|
| Accuracy on held-out validation set | Model doesn't generalize to new data |
| p95 latency < threshold | Model is too slow for real-time use |
| No NaN in outputs | Numerical instability that would break downstream systems |

Set criteria **before** running validation. Use `validate_model()` as the final checkpoint before any deployment script is triggered.


## Self-check

1. **What accuracy threshold would you set for a fraud detection model vs an Iris classifier?** Consider the cost of a false negative (missed fraud) vs a false positive (blocking a real customer) — and how that should change your threshold.
2. **Why check p95 latency rather than mean latency?** Look at the latency numbers printed above — how different are p50 and p95? Which number would users actually experience?
3. **What does it mean if the model passes accuracy tests but fails latency tests?** Think about what options you have: change the model architecture, optimize the preprocessing, or change the serving infrastructure.


## 📚 References

1. Breck, E., Cai, S., Nielsen, E., Salib, M., & Sculley, D. (2017). *The ML Test Score: A Rubric for ML Production Readiness and Technical Debt Reduction*. IEEE Big Data. <https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/>
2. Ribeiro, M. T., Wu, T., Guestrin, C., & Singh, S. (2020). *Beyond Accuracy: Behavioral Testing of NLP Models with CheckList*. ACL. <https://arxiv.org/abs/2005.04118>
3. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
